# Social Clubs Data Layer - Audit & Validation

**Branch:** social-clubs-alignment-audit  
**Referenz:** Issue #592

In [1]:
# Imports
import pandas as pd
import geopandas as gpd
from sqlalchemy import create_engine, text
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Database Connection
host = 'localhost'
port = 5433
database = 'layereddb'
user = 'alexander_kuhn'
password = 'aS5tY3uG1hM8vD6fP'

connection_string = f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{database}"
engine = create_engine(connection_string)

In [3]:
# Check available tables
query_tables = """
SELECT table_schema, table_name 
FROM information_schema.tables 
WHERE table_schema NOT IN ('pg_catalog', 'information_schema')
ORDER BY table_schema, table_name
"""
tables = pd.read_sql(query_tables, engine)
print("Available tables:")
print(tables)

Available tables:
           table_schema                   table_name
0         berlin_labels            district_features
1         berlin_labels          district_labels_new
2    berlin_recommender    district_level_aggregated
3    berlin_recommender  long_term_listings_features
4    berlin_source_data                        banks
..                  ...                          ...
184              public                      trigger
185              public                 unified_pois
186              public           unified_pois_stops
187              public                     variable
188              public                         xcom

[189 rows x 2 columns]


In [4]:
# Load Data - adjust table name if needed
TABLE_NAME = 'social_clubs_activities'  # Change if different
SCHEMA_NAME = 'berlin_source_data'       # Change if different

query = f"SELECT * FROM {SCHEMA_NAME}.{TABLE_NAME}"
df = pd.read_sql(query, engine)
print(f"Rows: {len(df)}")
print(f"\nColumns ({len(df.columns)}):")
print(df.columns.tolist())

Rows: 1658

Columns (22):
['id', 'name', 'club', 'leisure', 'sport', 'amenity', 'street', 'housenumber', 'postcode', 'website', 'phone', 'email', 'opening_hours', 'wheelchair', 'latitude', 'longitude', 'district', 'neighborhood_id', 'neighborhood', 'full_address', 'district_id', 'geometry']


In [5]:
# Show first rows
df.head()

,id,name,club,leisure,sport,amenity,street,housenumber,postcode,website,...,opening_hours,wheelchair,latitude,longitude,district,neighborhood_id,neighborhood,full_address,district_id,geometry
0,30012753,Umspannwerk,None,None,None,events_venue,None,None,None,None,...,unknown,true,52.494043,13.429187,Friedrichshain-Kreuzberg,0202,Kreuzberg,"Umspannwerk, Ohlauer Straße, Luisenstadt, Kreu...",11002002,POINT (13.4291868 52.4940425)
1,60775321,KW76,poker,None,None,None,Konrad-Wolf-Straße,76,None,None,...,unknown,unknown,52.538623,13.481623,Lichtenberg,1110,Alt-Hohenschönhausen,"KW76, 76, Konrad-Wolf-Straße, Wilhelmsberg, Al...",11011011,POINT (13.4816226 52.5386233)
2,257709121,Kulturhaus Spandau,None,None,None,arts_centre,None,None,None,None,...,unknown,true,52.535479,13.202312,Spandau,0501,Spandau,"Kulturhaus Spandau, Mauerstraße, Altstadt, Spa...",11005005,POINT (13.2023117 52.5354787)
3,266630320,Buergeramt Mahlsdorf,None,None,None,community_centre,Hönower Straße,91,12623,None,...,unknown,true,52.513140,13.612063,Marzahn-Hellersdorf,1004,Mahlsdorf,"Buergeramt Mahlsdorf, 91, Hönower Straße, Lich...",11010010,POINT (13.6120626 52.51314)
4,268915262,Karame e.V.,None,None,None,community_centre,Wilhelmshavener Straße,22,10551,None,...,"mo-fr 13:00-18:00; sa-su,ph off",false,52.531003,13.341544,Mitte,0102,Moabit,"Karame e.V., 22, Wilhelmshavener Straße, Alt-M...",11001001,POINT (13.3415438 52.5310025)


In [6]:
# Data types
print(df.dtypes)

id                  object
name                object
club                object
leisure             object
sport               object
amenity             object
street              object
housenumber         object
postcode            object
website             object
phone               object
email               object
opening_hours       object
wheelchair          object
latitude           float64
longitude          float64
district            object
neighborhood_id     object
neighborhood        object
full_address        object
district_id         object
geometry            object
dtype: object


## 1. Missing Values Analysis

In [7]:
# Missing Values Summary
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)

summary = pd.DataFrame({
    'Missing': missing,
    'Total': len(df),
    'Missing_Pct': missing_pct
}).sort_values('Missing_Pct', ascending=False)

print(summary)

                 Missing  Total  Missing_Pct
leisure             1596   1658        96.26
email               1485   1658        89.57
sport               1443   1658        87.03
phone               1379   1658        83.17
club                1313   1658        79.19
website              955   1658        57.60
postcode             602   1658        36.31
housenumber          563   1658        33.96
street               553   1658        33.35
amenity              302   1658        18.21
district               0   1658         0.00
district_id            0   1658         0.00
full_address           0   1658         0.00
neighborhood           0   1658         0.00
neighborhood_id        0   1658         0.00
id                     0   1658         0.00
longitude              0   1658         0.00
latitude               0   1658         0.00
wheelchair             0   1658         0.00
opening_hours          0   1658         0.00
name                   0   1658         0.00
geometry  

In [8]:
# Identify ID column (could be 'id', 'club_id', or other)
id_col = None
for col in ['id', 'club_id', 'ID', 'CLUB_ID']:
    if col in df.columns:
        id_col = col
        break

print(f"ID Column found: {id_col}")

# Define critical fields based on what exists
all_critical = ['id', 'club_id', 'name', 'latitude', 'longitude', 'geometry', 'district_id', 'neighborhood_id', 'district']
critical_fields = [f for f in all_critical if f in df.columns]

print(f"\nCritical Fields in table: {critical_fields}")

ID Column found: id

Critical Fields in table: ['id', 'name', 'latitude', 'longitude', 'geometry', 'district_id', 'neighborhood_id', 'district']


In [9]:
# Critical Fields Check
print("Critical Fields Missing Values:")
for field in critical_fields:
    null_count = df[field].isnull().sum()
    status = "OK" if null_count == 0 else "PROBLEM"
    print(f"  {field}: {null_count} ({status})")

Critical Fields Missing Values:
  id: 0 (OK)
  name: 0 (OK)
  latitude: 0 (OK)
  longitude: 0 (OK)
  geometry: 0 (OK)
  district_id: 0 (OK)
  neighborhood_id: 0 (OK)
  district: 0 (OK)


In [10]:
# OSM Tag Fields Analysis
osm_fields = [f for f in ['club', 'leisure', 'sport', 'amenity'] if f in df.columns]

print("OSM Tag Fields (high missing expected):")
for field in osm_fields:
    null_count = df[field].isnull().sum()
    pct = (null_count / len(df) * 100).round(1)
    print(f"  {field}: {null_count} missing ({pct}%)")

OSM Tag Fields (high missing expected):
  club: 1313 missing (79.2%)
  leisure: 1596 missing (96.3%)
  sport: 1443 missing (87.0%)
  amenity: 302 missing (18.2%)


In [11]:
# Check for 'unknown' placeholder values
print("Fields with 'unknown' placeholder:")
if 'opening_hours' in df.columns:
    print(f"  opening_hours = 'unknown': {(df['opening_hours'] == 'unknown').sum()}")
if 'wheelchair' in df.columns:
    print(f"  wheelchair = 'unknown': {(df['wheelchair'] == 'unknown').sum()}")

Fields with 'unknown' placeholder:
  opening_hours = 'unknown': 1289
  wheelchair = 'unknown': 1327


## 2. Data Validity Checks

In [12]:
# Primary Key Uniqueness
if id_col:
    duplicates = df[id_col].duplicated().sum()
    print(f"Duplicate {id_col}: {duplicates}")
    
    # numeric-only check
    non_numeric = df[~df[id_col].astype(str).str.match(r'^\d+$')]
    print(f"Non-numeric {id_col}: {len(non_numeric)}")
else:
    print("No ID column found!")

Duplicate id: 0
Non-numeric id: 0


In [13]:
# Unique values in categorical columns
if 'amenity' in df.columns:
    print("Unique amenity values:")
    print(df['amenity'].dropna().unique())

if 'wheelchair' in df.columns:
    print("\nUnique wheelchair values:")
    print(df['wheelchair'].unique())

Unique amenity values:
['events_venue' 'arts_centre' 'community_centre' 'social_centre' 'studio'
 'dojo' 'music_school' 'dancing_school' 'music_venue' 'photo_booth'
 'social_club']

Unique wheelchair values:
['true' 'unknown' 'false']


## 3. Geometry Validation

In [14]:
# Geometry Type Check
if 'geometry' in df.columns:
    df['geom_type'] = df['geometry'].apply(lambda x: 'POINT' if str(x).startswith('POINT') else 'OTHER' if pd.notna(x) else 'NULL')
    print("Geometry Types:")
    print(df['geom_type'].value_counts())
else:
    print("No geometry column found!")

Geometry Types:
geom_type
POINT    1658
Name: count, dtype: int64


In [15]:
# Coordinate Bounds (Berlin approx: Lat 52.34-52.68, Lon 13.09-13.76)
if 'latitude' in df.columns and 'longitude' in df.columns:
    print("Coordinate Ranges:")
    print(f"  Latitude:  {df['latitude'].min():.4f} to {df['latitude'].max():.4f}")
    print(f"  Longitude: {df['longitude'].min():.4f} to {df['longitude'].max():.4f}")

    # Points outside Berlin
    outside_berlin = df[
        (df['latitude'] < 52.3) | (df['latitude'] > 52.7) |
        (df['longitude'] < 13.0) | (df['longitude'] > 13.8)
    ]
    print(f"\nPoints outside Berlin bounds: {len(outside_berlin)}")
else:
    print("No lat/lon columns found!")

Coordinate Ranges:
  Latitude:  52.3739 to 52.6448
  Longitude: 13.1224 to 13.7311

Points outside Berlin bounds: 0


In [16]:
# Geometry Format Examples
if 'geometry' in df.columns:
    print("Geometry format examples:")
    print(df['geometry'].head(3).tolist())

Geometry format examples:
['POINT (13.4291868 52.4940425)', 'POINT (13.4816226 52.5386233)', 'POINT (13.2023117 52.5354787)']


## 4. District & Neighborhood ID Checks

In [17]:
# District ID Distribution
if 'district' in df.columns and 'district_id' in df.columns:
    print("District Distribution:")
    dist = df.groupby(['district', 'district_id']).size().reset_index(name='count').sort_values('count', ascending=False)
    print(dist)
else:
    print("District columns not found")
    print(f"Available columns: {df.columns.tolist()}")

District Distribution:
                      district district_id  count
4                        Mitte    11001001    247
1     Friedrichshain-Kreuzberg    11002002    215
6                       Pankow    11003003    177
11            Treptow-Köpenick    11009009    166
0   Charlottenburg-Wilmersdorf    11004004    152
5                     Neukölln    11008008    138
10        Tempelhof-Schöneberg    11007007    132
9          Steglitz-Zehlendorf    11006006    103
2                  Lichtenberg    11011011     95
7                Reinickendorf    11012012     91
8                      Spandau    11005005     81
3          Marzahn-Hellersdorf    11010010     61


In [18]:
# Check for orphan district_ids
try:
    query_orphans = f"""
    SELECT COUNT(*) as orphan_count
    FROM {SCHEMA_NAME}.{TABLE_NAME} sca
    LEFT JOIN {SCHEMA_NAME}.districts d ON sca.district_id = d.district_id
    WHERE d.district_id IS NULL
    """
    orphans = pd.read_sql(query_orphans, engine)
    print(f"Orphan district_ids: {orphans['orphan_count'].iloc[0]}")
except Exception as e:
    print(f"Could not check orphans: {e}")

Orphan district_ids: 0


In [19]:
# Neighborhood ID Check
if 'neighborhood_id' in df.columns and 'neighborhood' in df.columns:
    print("Neighborhood ID examples:")
    print(df[['neighborhood_id', 'neighborhood', 'district']].drop_duplicates().head(10))
else:
    print("Neighborhood columns not found")

Neighborhood ID examples:
   neighborhood_id          neighborhood                    district
0             0202             Kreuzberg    Friedrichshain-Kreuzberg
1             1110  Alt-Hohenschönhausen                 Lichtenberg
2             0501               Spandau                     Spandau
3             1004             Mahlsdorf         Marzahn-Hellersdorf
4             0102                Moabit                       Mitte
7             0506                Kladow                     Spandau
8             0701            Schöneberg        Tempelhof-Schöneberg
9             0401        Charlottenburg  Charlottenburg-Wilmersdorf
10            0106         Gesundbrunnen                       Mitte
11            0606            Nikolassee         Steglitz-Zehlendorf


In [20]:
# Check FK Constraint exists
try:
    query_constraints = f"""
    SELECT
        tc.constraint_name,
        kcu.column_name,
        ccu.table_name AS foreign_table,
        rc.delete_rule,
        rc.update_rule
    FROM information_schema.table_constraints tc
    JOIN information_schema.key_column_usage kcu ON tc.constraint_name = kcu.constraint_name
    JOIN information_schema.constraint_column_usage ccu ON ccu.constraint_name = tc.constraint_name
    JOIN information_schema.referential_constraints rc ON rc.constraint_name = tc.constraint_name
    WHERE tc.table_name = '{TABLE_NAME}'
      AND tc.constraint_type = 'FOREIGN KEY'
    """
    constraints = pd.read_sql(query_constraints, engine)
    print("Foreign Key Constraints:")
    if len(constraints) > 0:
        print(constraints)
    else:
        print("No FK constraints found!")
except Exception as e:
    print(f"Could not check constraints: {e}")

Foreign Key Constraints:
       constraint_name  column_name foreign_table delete_rule update_rule
0       district_id_fk  district_id     districts    RESTRICT     CASCADE
1       district_id_fk  district_id     districts    RESTRICT     CASCADE
2       district_id_fk  district_id     districts    RESTRICT     CASCADE
3       district_id_fk  district_id     districts    RESTRICT     CASCADE
4       district_id_fk  district_id     districts    RESTRICT     CASCADE
...                ...          ...           ...         ...         ...
101563  district_id_fk  district_id     districts    RESTRICT     CASCADE
101564  district_id_fk  district_id     districts    RESTRICT     CASCADE
101565  district_id_fk  district_id     districts    RESTRICT     CASCADE
101566  district_id_fk  district_id     districts    RESTRICT     CASCADE
101567  district_id_fk  district_id     districts    RESTRICT     CASCADE

[101568 rows x 5 columns]


## 5. Schema Check

In [21]:
# Current Schema
query_schema = f"""
SELECT column_name, data_type, character_maximum_length, is_nullable
FROM information_schema.columns
WHERE table_schema = '{SCHEMA_NAME}' 
  AND table_name = '{TABLE_NAME}'
ORDER BY ordinal_position
"""
schema = pd.read_sql(query_schema, engine)
print(schema)

        column_name          data_type  character_maximum_length is_nullable
0                id  character varying                     100.0          NO
1              name  character varying                     200.0          NO
2              club  character varying                     100.0         YES
3           leisure  character varying                     100.0         YES
4             sport  character varying                     100.0         YES
5           amenity  character varying                     100.0         YES
6            street  character varying                     200.0         YES
7       housenumber  character varying                      50.0         YES
8          postcode  character varying                      20.0         YES
9           website  character varying                     250.0         YES
10            phone  character varying                     100.0         YES
11            email  character varying                     150.0         YES

In [22]:
# Schema Compliance vs Issue #592
required_cols = ['id', 'district_id', 'name', 'latitude', 'longitude', 'geometry', 'neighborhood', 'district', 'neighborhood_id']
actual_cols = df.columns.tolist()

print("Schema Compliance Check:")
print("\nRequired columns:")
for col in required_cols:
    status = "OK" if col in actual_cols else "MISSING"
    print(f"  {col}: {status}")

print("\nExtra columns in table:")
extra = [c for c in actual_cols if c not in required_cols]
print(f"  {extra}")

Schema Compliance Check:

Required columns:
  id: OK
  district_id: OK
  name: OK
  latitude: OK
  longitude: OK
  geometry: OK
  neighborhood: OK
  district: OK
  neighborhood_id: OK

Extra columns in table:
  ['club', 'leisure', 'sport', 'amenity', 'street', 'housenumber', 'postcode', 'website', 'phone', 'email', 'opening_hours', 'wheelchair', 'full_address', 'geom_type']


## 6. Summary Report

In [23]:
# Final Summary
print("="*50)
print("AUDIT SUMMARY")
print("="*50)
print(f"Table: {SCHEMA_NAME}.{TABLE_NAME}")
print(f"Total Records: {len(df)}")
print(f"Total Columns: {len(df.columns)}")
print(f"\nColumns: {df.columns.tolist()}")
print("="*50)

AUDIT SUMMARY
Table: berlin_source_data.social_clubs_activities
Total Records: 1658
Total Columns: 23

Columns: ['id', 'name', 'club', 'leisure', 'sport', 'amenity', 'street', 'housenumber', 'postcode', 'website', 'phone', 'email', 'opening_hours', 'wheelchair', 'latitude', 'longitude', 'district', 'neighborhood_id', 'neighborhood', 'full_address', 'district_id', 'geometry', 'geom_type']


In [24]:
# Cleanup temp columns
if 'geom_type' in df.columns:
    df.drop(columns=['geom_type'], inplace=True)